# Projekt VBridge – BML

Petr Matoušů



## 1. Import knihoven


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge, BayesianRidge
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# Pro reproducibilitu
np.random.seed(42)


## 2. Načtení a předzpracování dat

Použijeme dataset California Housing od sklearnu.  
- Vlastnosti: 8 numerických proměnných, cílová proměnná `MedHouseVal`.  
- Rozdělíme na trénovací a testovací s poměrem 80/20.


In [2]:
# Načtení dat
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

# Rozdělení
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Trénovacích vzorků:", X_train.shape[0])
print("Testovacích vzorků:",  X_test.shape[0])


Trénovacích vzorků: 16512
Testovacích vzorků: 4128


## 3. Klasická ridge regrese

1. Prohledáme hyperparametr `alpha` pomocí `GridSearchCV`.  
2. Vyhodnotíme MSE a $(R^2)$ na testovací sadě.  


In [3]:
# Grid search pro klasickou ridge regresi
param_grid = {'alpha': np.logspace(-3, 3, 13)}
ridge_cv = GridSearchCV(Ridge(), param_grid, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train, y_train)

best_alpha = ridge_cv.best_params_['alpha']
print(f"Nejlepší alpha (Ridge): {best_alpha:.4f}")

# Trénink s nejlepším alpha
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train, y_train)

# Predikce a vyhodnocení
y_pred_ridge = ridge.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge  = r2_score(y_test, y_pred_ridge)

print(f"Ridge Regression – MSE: {mse_ridge:.4f}, R2: {r2_ridge:.4f}")


Nejlepší alpha (Ridge): 10.0000
Ridge Regression – MSE: 0.5550, R2: 0.5764


- **Nejlepší $\alpha$**: 10.0  
- **Testovací MSE**: 0.5550  
- **Testovací $R^2$**: 0.5764  

Model s $\alpha = 10$ dosahuje solidní predikční přesnosti na datech California Housing. Hodnota $R^2 \approx 0.58$  znamená, že asi 58 % variability ceny domů je vysvětleno lineárním modelem s regularizací. MSE na úrovni 0.5550 ukazuje, že průměrná kvadratická chyba predikce je přibližně $\sqrt{0.5550} \approx 0.745$ v jednotkách mediánové ceny.
## 4. Bayesovská ridge regrese (sklearn)

1. `BayesianRidge` automaticky odhaduje `alpha` (prior precision) a `lambda_` (noise precision) z dat.  
2. Vyhodnocení na testovací sadě.


In [4]:
bayesian = BayesianRidge()
bayesian.fit(X_train, y_train)

# Posterior mean koeficientů
coef_bayes = bayesian.coef_
print("Posterior mean koeficientů (Bayes.):\n", coef_bayes)

# Predikce a vyhodnocení
y_pred_bayes = bayesian.predict(X_test)
mse_bayes = mean_squared_error(y_test, y_pred_bayes)
r2_bayes  = r2_score(y_test, y_pred_bayes)

print(f"Bayesian Ridge – MSE: {mse_bayes:.4f}, R2: {r2_bayes:.4f}")


Posterior mean koeficientů (Bayes.):
 [ 4.48104693e-01  9.73022717e-03 -1.22248093e-01  7.77134182e-01
 -2.01627501e-06 -3.52472317e-03 -4.19771337e-01 -4.33611354e-01]
Bayesian Ridge – MSE: 0.5556, R2: 0.5760


### Diskuze výsledků

- **Posterior mean koeficientů**:  
  $[+0.4481,\; +0.0097,\; -0.1222,\; +0.7771,\; -0.000002,\; -0.0035,\; -0.4198,\; -0.4336]$  
- **Testovací MSE**: 0.5556  
- **Testovací $R^2$**: 0.5760  

Bayesovská ridge regrese dosahuje téměř stejné přesnosti jako klasická ridge (rozdíl v MSE jen +0.0006, v $(R^2)$ −0.0004). Navíc poskytuje posteriorní rozdělení koeficientů, což umožňuje kvantifikovat nejistotu odhadů. Koeficienty jsou v průměru velmi podobné těm z klasické verze, s drobnými odchylkami v řádu desetin procenta.


## 5. Bayesovská regrese se „pozitivity prior“

Zavedeme předpoklad positivity tak, že místo nultého prior-mean použijeme kladný prior-mean $m_0 = c \mathbf{1}$.  
Posterior pak je:
$
\Sigma_N = \bigl(\alpha I + \beta X^\top X\bigr)^{-1},\quad
m_N = \Sigma_N \bigl(\beta X^\top y + \alpha m_0\bigr).
$
- $\beta = 1/\mathrm{Var}(y - Xw)$ odhadneme z Bayes. ridge
- $\alpha =$ odpovídá BayesianRidge.alpha_

Vyzkoušíme s \(c=1\).


In [5]:
# Parametry z předchozí Bayes. regrese
alpha_hat = bayesian.alpha_     # prior precision
beta_hat  = bayesian.lambda_    # noise precision

# Prior mean (všechny koeficienty = 1)
m0 = np.ones(X_train.shape[1])

# Posterior covariance a mean
Sigma_N = np.linalg.inv(alpha_hat * np.eye(X_train.shape[1]) + beta_hat * X_train.T.dot(X_train))
m_N     = Sigma_N.dot(beta_hat * X_train.T.dot(y_train) + alpha_hat * m0)

print("Posterior mean s positivity prior:\n", m_N)

# Predikce (map)
y_pred_pos = X_test.dot(m_N)
mse_pos = mean_squared_error(y_test, y_pred_pos)
r2_pos  = r2_score(y_test, y_pred_pos)

print(f"Bayes w/ positivity prior – MSE: {mse_pos:.4f}, R2: {r2_pos:.4f}")


Posterior mean s positivity prior:
 [ 5.22526717e-01  1.58931037e-02 -1.93875195e-01  9.75432954e-01
  9.22500351e-06 -4.40382831e-03 -6.21567656e-02 -1.49823362e-02]
Bayes w/ positivity prior – MSE: 0.6294, R2: 0.5197


### Diskuze positivity prior

- **Posterior mean koeficientů s \(m_0 = \mathbf{1}\)**:  
  $[+0.5225,\; +0.0159,\; -0.1939,\; +0.9754,\; +0.000009,\; -0.0044,\; -0.0622,\; -0.01498]$  
- **Testovací MSE**: 0.6294  
- **Testovací \(R^2\)**: 0.5197  

Zavedení kladného pri­oru mean $m_0=1$ posunulo některé koeficienty směrem vzhůru (zejména čtvrtá proměnná), avšak na úkor predikční přesnosti: MSE vzrostla o ~13 % a $R^2$ kleslo o ~0.056. Tento předpoklad je vhodný jen tehdy, pokud z doménových znalostí opravdu víme, že všechny efekty musí být kladné. V opačném případě zavádí výrazné zkreslení.



## 6. Závěr

1. **Klasická ridge regrese**  
   - Tuning $\alpha$ přinesl MSE 0.5550 a $R^2$ 0.5764.  

2. **Bayesovská ridge regrese**  
   - Velmi podobné výsledky: MSE 0.5556, $R^2$ 0.5760.  
   - Navíc umožňuje odhad nejistoty koeficientů.  

3. **Bayesovská regrese s positivity prior**  
   - Zavedení kladného prior-mean $m_0=1$ zvýšilo MSE na 0.6294 a snížilo $R^2$ na 0.5197.  
   - Užitečné, pokud víme, že všechny koeficienty musí být kladné; jinak riziko systematického zkreslení.

**Doporučení:**  
– Pro obecnou regresi bez silné doménové znalosti zůstat u klasické nebo Bayesovské ridge regrese.  
– Pokud je validní předpoklad positivity, lze zvolit positivity prior, ale je třeba být opatrný na dopad na přesnost.